# General Table Extraction

Extracts tables from scanned PDF pages using Gemini vision LLM.

Uses modules from `agents.general_table` and `shared`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Ensure project root is on sys.path so imports work
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.client import client, DEFAULT_MODEL
from shared.pdf import render_pages
from shared.eval import compare_tables, print_accuracy_summary
from agents.general_table.extract import extract_tables_from_page, stitch_page_results

print(f"Client ready, model: {DEFAULT_MODEL}")

In [ ]:
# Configuration
PDF_PATH = str(PROJECT_ROOT / "example sources" / "CPY Document.pdf")
PAGES = [31]  # 1-indexed
MODEL = DEFAULT_MODEL
DPI = 300
OUTPUT_DIR = Path("test_data") / "output"
GROUND_TRUTH_CSV = Path("test_data") / "ground_truth" / "principal_investigators.csv"

print(f"PDF: {PDF_PATH}")
print(f"Pages: {PAGES}")
print(f"Model: {MODEL}")

In [ ]:
# Render and display pages
page_images = render_pages(PDF_PATH, PAGES, dpi=DPI)

for page_num, img in page_images.items():
    print(f"Page {page_num}: {img.size[0]}x{img.size[1]} pixels")
    display_width = 600
    ratio = display_width / img.size[0]
    display(img.resize((display_width, int(img.size[1] * ratio))))

In [ ]:
# Extract tables from each page
page_results = {}
continuation_headers = None

for page_num in PAGES:
    print(f"Processing page {page_num} (continuation={'yes' if continuation_headers else 'no'})...")
    result = extract_tables_from_page(
        client, MODEL, page_images[page_num], continuation_headers=continuation_headers
    )
    page_results[page_num] = result

    for i, table in enumerate(result.tables):
        label = "continuation" if not table.headers else ", ".join(table.headers[:3]) + ("..." if len(table.headers) > 3 else "")
        print(f"  Table {i+1}: {len(table.rows)} rows | headers: {label} | complete: {table.table_appears_complete}")

    if result.tables and not result.tables[-1].table_appears_complete:
        last = result.tables[-1]
        continuation_headers = last.headers if last.headers else continuation_headers
    else:
        continuation_headers = None

In [ ]:
# Stitch and preview results
extracted_tables = stitch_page_results(page_results)
print(f"Total logical tables: {len(extracted_tables)}\n")

for i, df in enumerate(extracted_tables):
    print(f"Table {i+1}: {len(df)} rows x {len(df.columns)} columns — {list(df.columns)}")
    display(df)
    print()

In [ ]:
# Save to CSV
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for i, df in enumerate(extracted_tables):
    filename = OUTPUT_DIR / f"gemini_table_{i+1}.csv"
    df.to_csv(filename, index=False)
    print(f"Saved: {filename} ({len(df)} rows x {len(df.columns)} cols)")

In [ ]:
# Evaluate against ground truth
if GROUND_TRUTH_CSV.exists():
    TABLE_INDEX = 0
    df_truth = pd.read_csv(GROUND_TRUTH_CSV, dtype=str).fillna("")
    df_extracted = pd.read_csv(OUTPUT_DIR / f"gemini_table_{TABLE_INDEX+1}.csv", dtype=str).fillna("")

    print(f"Ground truth: {len(df_truth)} rows x {len(df_truth.columns)} cols")
    print(f"Extracted:    {len(df_extracted)} rows x {len(df_extracted.columns)} cols\n")

    results = compare_tables(df_truth, df_extracted)
    print_accuracy_summary(results)
else:
    print(f"Ground truth not found at {GROUND_TRUTH_CSV} — skipping eval.")